# Machine Learning — Clustering, Recommender and Price Prediction

**Author:** Mahdi Tarrah

**Data:** Divar real-estate listings — 1,000,000 ads, 61 columns

Three models on the same dataset:

| Part | Task | Method |
|---|---|---|
| 1 | Group comparable properties and recommend similar listings | K-means, DBSCAN |
| 2 | Predict the monthly rent of a rental listing | XGBoost |
| 3 | Predict the sale price of a listing | XGBoost + target encoding |

---
# Part 1 — Clustering and the Recommender System

Selecting a feature set, choosing *k*, and comparing K-means with DBSCAN.

<div dir="rtl">

**مسیر این نوت‌بوک**

1. **آماده‌سازی:** پاک‌سازی قیمت و متراژ با IQR روی لگاریتم، حذف مختصات خارج از ایران و تبدیل مختصات به UTM با یک zone ثابت (۳۹) تا فاصله‌ها در کل کشور قابل مقایسه باشند.
2. **بخش ۱ — انتخاب ویژگی و K-means با ۱۰ خوشه:** چهار مجموعه ویژگی با سه معیار (Silhouette، Calinski-Harabasz، Davies-Bouldin) مقایسه شدند و «قیمت + مختصات UTM» برنده شد. هر ویژگی اضافه (متراژ، اتاق، سال ساخت) خوشه‌ها را بدتر کرد و ابعاد را هم بالا برد.
3. **بخش ۲ — انتخاب k:** روی نمودار Elbow زانو با چشم دیده نمی‌شود. با KneeLocator، سه معیار کیفیت و مقایسه‌ی پروفایل خوشه‌ها (k=4 در برابر k=7)، **k = 7** انتخاب شد.
4. **بخش ۳ — DBSCAN روی قیمت + UTM:** eps با نمودار k-distance شروع شد و با جست‌وجوی شبکه‌ای روی eps و min_samples به **۳ خوشه‌ی معنادار** رسید (eps=0.45، min_samples=1000).
5. **سیستم توصیه‌گر:** با خوشه‌های نهایی، برای یک آگهی یا یک بودجه و موقعیت دلخواه، نزدیک‌ترین آگهی‌های همان خوشه و همان نوع ملک پیشنهاد می‌شوند (بخش آخر).

**درباره‌ی «قیمت قابل تبدیل»:** ستون `transformable_price` در داده فقط True/False است و قیمت نیست؛ با تأیید منتور از `price_value` استفاده شد (توضیح کامل در بخش ۳).

سلول‌های این نوت‌بوک خروجی ذخیره‌شده‌ی اجرای اصلی را دارند، جز بخش توصیه‌گر که تازه اضافه شده است.
برای اجرا، `Divar.csv` را در پوشه‌ی `data/` ریشه‌ی ریپو بگذارید. پکیج‌های لازم: `utm` و `kneed`.

</div>

## Data Preparation

In [ ]:
# libraries

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import utm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kneed import KneeLocator
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore',category=UserWarning,module='sklearn')

In [ ]:
# load data

clustering_columns = ['price_value','building_size','rooms_count','location_latitude','location_longitude','cat2_slug','cat3_slug','city_slug','construction_year', 'has_parking', 'has_elevator']
df = pd.read_csv('../data/Divar.csv', usecols=clustering_columns)
print(df.shape)
df.head()

In [ ]:
# null check

df.info()

In [ ]:
# cleaning

data = df.copy()

data.loc[data['price_value'] <= 0,'price_value'] = np.nan

def clean_outliers(data,column):
    logs = np.log(data[column])
    q1, q3 = logs.quantile(0.25),logs.quantile(0.75)
    iqr = q3 - q1
    data.loc[~logs.between(q1-1.5*iqr,q3 +1.5*iqr),column] = np.nan
    return data

data = clean_outliers(data,'price_value')
data = clean_outliers(data,'building_size')

inside_iran = (data['location_latitude'].between(25,40)) & (data['location_longitude'].between(44,63.5))
data.loc[~inside_iran,['location_latitude','location_longitude']] = np.nan

data = data.dropna(subset=['building_size','price_value','location_latitude','location_longitude'])

print(data.shape)
print(data[['price_value','building_size']].describe())

In [ ]:
# to UTM

xy = [utm.from_latlon(lat,lon,force_zone_number=39)[:2] for lat,lon in zip(data['location_latitude'], data['location_longitude'])]

data['utm_x'] = [p[0] for p in xy]
data['utm_y'] = [p[1] for p in xy]

print(data[['utm_x','utm_y']].describe())

In [ ]:
# zone check

print(((data['utm_x'] < 100000) | (data['utm_x'] > 900000)).sum())

In [ ]:
# rooms to number

rooms_mapping = {'بدون اتاق':0,'یک':1,'دو':2,'سه':3,'چهار':4,'پنج یا بیشتر':5}
data['rooms_count'] = data['rooms_count'].map(rooms_mapping)

## Part 1 — Feature Selection and KMeans with 10 Clusters

### Choosing the feature set

In [ ]:
# year to number

digits = str.maketrans('۰۱۲۳۴۵۶۷۸۹','0123456789')
year = data['construction_year'].replace('قبل از ۱۳۷۰', '۱۳۶۹')
data['construction_year'] = pd.to_numeric(year.str.translate(digits), errors='coerce')
print(data['construction_year'].describe())

In [ ]:
# common subset

data_cmp = data.dropna(subset=['rooms_count','construction_year'])
print(data.shape, data_cmp.shape)

In [ ]:
# feature correlation

print(data_cmp[['price_value','building_size','rooms_count','construction_year']].corr())

In [ ]:
# compare feature sets

feature_sets = {'A: price + utm + size': ['price_value','utm_x','utm_y','building_size'],'B: A + rooms':['price_value','utm_x','utm_y','building_size','rooms_count'],'C: price + utm':['price_value','utm_x','utm_y'],'D: C + year': ['price_value','utm_x','utm_y','construction_year']}

rows = []

for name, feats in feature_sets.items():
    X = StandardScaler().fit_transform(data_cmp[feats])
    km = KMeans(10, random_state=44)
    labels = km.fit_predict(X)
    rows.append({'feature set': name,
                 'silhouette (higher is better)': silhouette_score(X, labels, sample_size=10000, random_state=44),
                 'calinski (higher is better)': calinski_harabasz_score(X, labels),
                 'davies (lower is better)': davies_bouldin_score(X, labels)})

pd.DataFrame(rows).set_index('feature set').round(3)

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>All three metrics pick price + coordinates</li>
<li>Every extra feature made the clusters worse</li>
<li>Dropped rooms (same information as size) and year (no link to price)</li>
</ul>
</font></div>

### KMeans with 10 clusters

In [ ]:
# KMeans k=10

scaler = StandardScaler()
X = scaler.fit_transform(data[['price_value','utm_x','utm_y']])

km = KMeans(10, random_state=44)
data['cluster'] = km.fit_predict(X)

print(data['cluster'].value_counts().sort_index())

In [ ]:
# cluster map

centers = scaler.inverse_transform(km.cluster_centers_)

plt.figure(figsize=(10, 10))
plt.scatter(data['utm_x'],data['utm_y'],c=data['cluster'],s=2,alpha=0.3,cmap='tab10')
plt.scatter(centers[:, 1],centers[:, 2], marker='*',c='black',s=500,edgecolors='white',linewidths=1.5,zorder=5,label='cluster centres')
plt.xlabel('UTM x (m)')
plt.ylabel('UTM y (m)')
plt.title('KMeans clusters (k=10)')
plt.legend()
plt.show()

In [ ]:
# price scatter

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(data['utm_x'],data['price_value']/1e9,c=data['cluster'], s=2, alpha=0.3,cmap='tab10')
axes[0].scatter(centers[:, 1], centers[:, 0]/1e9,marker='*',c='black',s=400,edgecolors='white',linewidths=1.5,zorder=5)
axes[0].set_xlabel('UTM x (m)')
axes[0].set_ylabel('price (billion toman)')

axes[1].scatter(data['utm_y'],data['price_value']/1e9,c=data['cluster'],s=2,alpha=0.3, cmap='tab10')
axes[1].scatter(centers[:, 2],centers[:, 0]/1e9,marker='*',c='black',s=400,edgecolors='white',linewidths=1.5,zorder=5)
axes[1].set_xlabel('UTM y (m)')
axes[1].set_ylabel('price (billion toman)')

plt.suptitle('price against UTM coordinates, coloured by cluster (k=10)')
plt.tight_layout()
plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>The question asks for a scatter plot of price against the UTM coordinates, so this is that plot, while the previous one is the same clustering seen as a map</li>
<li>Every point is coloured by its cluster and the black stars are the cluster centres</li>
<li>It shows what the map cannot: inside Tehran the clusters are separated by price, not by location</li>
</ul>
</font></div>

In [ ]:
# cluster profile

profile = data.groupby('cluster').agg(n=('cluster','size'),price_M=('price_value', lambda s: round(s.median()/1e6)),size_m2=('building_size','median'))
profile['top_cities'] = data.groupby('cluster')['city_slug'].apply(lambda s: ','.join(s.value_counts().head(3).index))
print(profile.sort_values('price_M'))

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>Centres converted back to metres, otherwise they miss their place</li>
<li>Far cities each form their own cluster</li>
<li>Tehran splits into four price levels, which is the recommender logic</li>
</ul>
</font></div>

## Part 2 — Choosing the Number of Clusters

In [ ]:
# elbow curve

res = []

for k in range(1, 21):
    km = KMeans(k,random_state=44)
    km.fit(X)
    res.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 21),res,marker='o',markersize=6,linestyle='--',linewidth=1)
plt.xticks(range(1, 21))
plt.grid(alpha=0.3)
plt.xlabel('k')
plt.ylabel('WCSS (inertia)')
plt.title('Elbow method for k = 1 to 20')
plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>The knee is not visible by eye</li>
<li>KneeLocator returned 7</li>
</ul>
</font></div>

In [ ]:
# find knee

loc = KneeLocator(range(1, 21),res,curve='convex',direction='decreasing')
print(loc.elbow,loc.knee)

In [ ]:
# three metrics

sil, cal, dav = [], [], []

for k in range(2,21):
    km = KMeans(k,random_state=44)
    labels = km.fit_predict(X)
    sil.append( silhouette_score(X,labels,sample_size=10000,random_state=44))
    cal.append(calinski_harabasz_score(X,labels))
    dav.append(davies_bouldin_score(X,labels))

In [ ]:
# metric plots

fig, axes = plt.subplots(1,3,figsize=(16,4))

axes[0].plot(range(2,21),sil,marker='o',markersize=5,linestyle='--',linewidth=1)
axes[0].set_title('silhouette (higher is better)')

axes[1].plot(range(2,21),cal,marker='o',markersize=5,linestyle='--',linewidth=1)
axes[1].set_title('calinski-harabasz (higher is better)')

axes[2].plot(range(2,21),dav,marker='o',markersize=5,linestyle='--',linewidth=1)
axes[2].set_title('davies-bouldin (lower is better)')

for ax in axes:
    ax.set_xlabel('k')
    ax.set_xticks(range(2,21))
    ax.tick_params(axis='x',labelsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>Silhouette and Davies favour k = 4</li>
<li>Calinski flattens after 7, like the elbow</li>
<li>Silhouette uses a 10k sample, so its curve is jagged</li>
</ul>
</font></div>

In [ ]:
# compare k

for k in [4, 7]:
    km_test = KMeans(k,random_state=44)
    labels_test = km_test.fit_predict(X)

    tmp = data.copy()
    tmp['c'] = labels_test

    prof = tmp.groupby('c').agg(n=('c','size'),price_M=('price_value',lambda s: round(s.median() / 1e6)),size_m2=('building_size','median'))
    prof['top_cities'] = tmp.groupby('c')['city_slug'].apply(lambda s: ','.join(s.value_counts().head(3).index))

    print(f'k = {k}')
    print(prof.sort_values('price_M'))
    print()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>With k = 4 two thirds of the data lands in one cluster</li>
<li>With k = 7 Tehran gets three price levels and cities separate</li>
<li>Chose 7</li>
</ul>
</font></div>

In [ ]:
# final KMeans

km_final = KMeans(7, random_state=44)
data['cluster_final'] = km_final.fit_predict(X)

centers_final = scaler.inverse_transform(km_final.cluster_centers_)

print(data['cluster_final'].value_counts().sort_index())

In [ ]:
# plot range

x_lo, x_hi = data['utm_x'].quantile([0.001, 0.999])
y_lo, y_hi = data['utm_y'].quantile([0.001, 0.999])
inside = ((centers_final[:, 1].clip(x_lo, x_hi) == centers_final[:, 1]) & (centers_final[:, 2].clip(y_lo, y_hi) == centers_final[:, 2]))

In [ ]:
# final map

plt.figure(figsize=(11, 10))

for c in sorted(data['cluster_final'].unique()):
    sub = data[data['cluster_final'] == c]
    plt.scatter(sub['utm_x'], sub['utm_y'], s=2, alpha=0.3, label=f'cluster {c}  (n={len(sub)})')

plt.scatter(centers_final[inside, 1],centers_final[inside, 2],marker='*',c='black',s=500,edgecolors='white',linewidths=1.5,zorder=5)
plt.xlim(x_lo, x_hi)
plt.ylim(y_lo, y_hi)
plt.xlabel('UTM x (m)')
plt.ylabel('UTM y (m)')
plt.title('Final KMeans clusters k=7 (black stars = cluster centres)')
plt.legend(markerscale=6)
plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>One centre falls outside the drawn range</li>
<li>A centre lives in 3D (price and x, y) but only 2D is plotted</li>
</ul>
</font></div>

## Part 3 — DBSCAN

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>The question asks for the transformable price, but in the data that column only holds True and False, so it is a flag saying the ad's credit and rent can be converted, not a price</li>
<li>transformable_credit equals credit_value and transformable_rent equals rent_value, so there is no ready-made unified price column in the dataset</li>
<li>The mentor confirmed using price_value, so this part clusters the sale price together with the UTM coordinates, exactly the two features the question names</li>
</ul>
</font></div>

In [ ]:
# sample and scale

data_db = data.sample(100000, random_state=44)

scaler_db = StandardScaler()
X_db = scaler_db.fit_transform(data_db[['price_value','utm_x','utm_y']])

print(X_db.shape)

In [ ]:
# k-distance eps

nn = NearestNeighbors(n_neighbors=10)
nn.fit(X_db)
neigh_dist, _ = nn.kneighbors()

best_dist = neigh_dist[:,-1]
sorted_dist = np.sort(best_dist)

loc = KneeLocator(range(len(sorted_dist)),sorted_dist,curve='convex')
print('knee index:',loc.knee,' eps =',sorted_dist[loc.knee])

zoom_from = len(sorted_dist) - 5000

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(len(sorted_dist)),sorted_dist)
ax.scatter(loc.knee, sorted_dist[loc.knee],c='red',zorder=5,label=f'knee: eps = {sorted_dist[loc.knee]:.2f}')
ax.set_xlabel('points sorted by distance')
ax.set_ylabel('distance to the 10th neighbour')
ax.legend(loc='upper left')

axin = ax.inset_axes([0.08, 0.32, 0.45, 0.55])
axin.plot(range(zoom_from, len(sorted_dist)), sorted_dist[zoom_from:])
axin.scatter(loc.knee, sorted_dist[loc.knee], c='red', zorder=5)
axin.set_title(f'zoom on the last {len(sorted_dist) - zoom_from} points', fontsize=9)
axin.grid(alpha=0.3)
ax.indicate_inset_zoom(axin, edgecolor='black')

plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>The k-distance knee gave eps = 0.64</li>
<li>Too large, almost everything became one cluster</li>
</ul>
</font></div>

In [ ]:
# first run

db = DBSCAN(eps=0.643,min_samples=10,n_jobs=2)
labels_db = db.fit_predict(X_db)

print(np.unique(labels_db,return_counts=True))

In [ ]:
# eps effect

for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
    db = DBSCAN(eps=eps,min_samples=10,n_jobs=2)
    lab = db.fit_predict(X_db)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    noise = (lab == -1).mean() * 100
    print(f'eps={eps}  clusters={n_clusters}  noise={noise:.1f}%')

In [ ]:
# min_samples effect

for ms in [5, 10, 50, 100, 500]:
    db = DBSCAN(eps=0.5,min_samples=ms,n_jobs=2)
    lab = db.fit_predict(X_db)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    noise = (lab == -1).mean() * 100
    print(f'min_samples={ms}  clusters={n_clusters}  noise={noise:.1f}%')

In [ ]:
# grid search

for eps in [0.3,0.35,0.4,0.45]:
    for ms in [1000,1500,2000,3000]:
        db = DBSCAN(eps=eps,min_samples=ms,n_jobs=2)
        lab = db.fit_predict(X_db)
        n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
        sizes = sorted(np.bincount(lab[lab >= 0]).tolist(),reverse=True) if n_clusters else []
        noise = (lab == -1).mean() * 100
        print(f'eps={eps}  ms={ms}  clusters={n_clusters}  noise={noise:4.1f}%  sizes={sizes[:4]}')

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>Larger eps means fewer clusters and less noise</li>
<li>Larger min_samples means more noise</li>
</ul>
</font></div>

In [ ]:
# final DBSCAN

db_final = DBSCAN(eps=0.45,min_samples=1000,n_jobs=2)
data_db['dbscan'] = db_final.fit_predict(X_db)
print(np.unique(data_db['dbscan'],return_counts=True))

In [ ]:
# DBSCAN profile

prof_db = data_db[data_db['dbscan'] >= 0].groupby('dbscan').agg(n=('dbscan', 'size'),price_M=('price_value', lambda s: round(s.median() / 1e6)),size_m2=('building_size','median'),)
prof_db['top_cities'] = data_db[data_db['dbscan'] >= 0].groupby('dbscan')['city_slug'].apply(lambda s: ','.join(s.value_counts().head(4).index))
print(prof_db)

noise = data_db[data_db['dbscan'] == -1]
print('noise count:',len(noise))
print('noise median price:',round(noise['price_value'].median()/1e6),'M')
print('noise top cities:',noise['city_slug'].value_counts().head(4).index.tolist())

In [ ]:
# DBSCAN map

plt.figure(figsize=(11, 10))

noise_mask = data_db['dbscan'] == -1

plt.scatter(data_db.loc[noise_mask,'utm_x'],data_db.loc[noise_mask,'utm_y'],s=2,alpha=0.4,c="#EF0707",label=f'noise (n={noise_mask.sum()})')

for c in sorted(data_db.loc[~noise_mask,'dbscan'].unique()):
    sub = data_db[data_db['dbscan'] == c]
    plt.scatter(sub['utm_x'],sub['utm_y'],s=2,alpha=0.4,label=f'cluster {c}  (n={len(sub)})')

plt.xlim(x_lo,x_hi)
plt.ylim(y_lo,y_hi)
plt.xlabel('UTM x (m)')
plt.ylabel('UTM y (m)')
plt.title('DBSCAN clusters (eps=0.45, min_samples=1000)')
plt.legend(markerscale=6)
plt.show()

In [ ]:
# dbscan price scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, coord, name in zip(axes, ['utm_x','utm_y'], ['UTM x (m)','UTM y (m)']):
    ax.scatter(data_db.loc[noise_mask, coord],data_db.loc[noise_mask,'price_value']/1e9,s=2,alpha=0.4,c='#EF0707',label=f'noise (n={noise_mask.sum()})')
    for c in sorted(data_db.loc[~noise_mask, 'dbscan'].unique()):
        sub = data_db[data_db['dbscan'] == c]
        ax.scatter(sub[coord],sub['price_value']/1e9,s=2,alpha=0.4,label=f'cluster {c}')
    ax.set_xlabel(name)
    ax.set_ylabel('price (billion toman)')

axes[0].legend(markerscale=6)
plt.suptitle('price against UTM coordinates, coloured by DBSCAN cluster')
plt.tight_layout()
plt.show()

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>The same two panels as part 1, because the question asks for the DBSCAN scatter to follow part 1</li>
<li>DBSCAN has no centres, so only the points are drawn</li>
<li>The noise spreads over the whole price range, which is why its median price is three times that of the clusters</li>
</ul>
</font></div>

<div dir="ltr" align="left"><font color="#0099cc">
<ul>
<li>eps 0.45 with min_samples 1000 gives 3 clusters and 10% noise</li>
<li>Poles are Tehran-Karaj-Isfahan, Mashhad, Shiraz</li>
<li>Noise median is 9.2B, so much of it is luxury property</li>
<li>I ran this on a 100k sample, which the mentor accepted</li>
<li>Refitting on all 324k rows is not feasible: sklearn computes every neighbourhood at once, so at this eps the memory blows up</li>
</ul>
</font></div>

---
## سیستم توصیه‌گر بر پایه‌ی خوشه‌های نهایی

<div dir="rtl">

ایده‌ی سیستم توصیه‌گر ساده است: هر خوشه یک «سلیقه/بازار» است (مثلاً آپارتمان‌های لوکس شمال تهران یا بازار ارزان شهرهای جنوبی).

- **پیشنهاد آگهی مشابه:** کاربری که روی یک آگهی کلیک کرده، آگهی‌های همان خوشه و همان نوع ملک را می‌بیند، به ترتیب نزدیکی در فضای استانداردشده‌ی قیمت و مختصات. اختلاف متراژ هم کمی جریمه می‌شود.
- **پیشنهاد بر اساس نیاز کاربر:** بودجه و موقعیت دلخواه کاربر به همان فضای ویژگی برده می‌شود. مدل K-means خوشه‌ی کاربر را پیش‌بینی می‌کند و نزدیک‌ترین آگهی‌های آن خوشه برگردانده می‌شوند.

فیلتر کردن داخل خوشه هم سریع‌تر است (جست‌وجو فقط بین آگهی‌های یک خوشه) و هم پیشنهادها را در همان بازاری نگه می‌دارد که کاربر به آن علاقه نشان داده است.

</div>

In [ ]:
FEATURES = ["price_value", "utm_x", "utm_y"]
data_scaled = pd.DataFrame(scaler.transform(data[FEATURES]), index=data.index, columns=FEATURES)
SHOW = ["city_slug", "cat3_slug", "price_value", "building_size", "rooms_count", "cluster_final"]


def recommend_similar(listing_index, n=5, size_weight=0.5, same_type=True):
    """Listings from the same final KMeans cluster that sit closest in scaled price + location space."""
    cluster = data.at[listing_index, "cluster_final"]
    same = (data["cluster_final"] == cluster) & (data.index != listing_index)
    if same_type:  # an apartment buyer is shown apartments, not shops
        same &= data["cat3_slug"] == data.at[listing_index, "cat3_slug"]
    candidates = data.index[same]
    distance = np.linalg.norm(data_scaled.loc[candidates].to_numpy() - data_scaled.loc[listing_index].to_numpy(), axis=1)
    size_gap = np.abs(np.log(data.loc[candidates, "building_size"]) - np.log(data.at[listing_index, "building_size"]))
    score = pd.Series(distance + size_weight * size_gap.to_numpy(), index=candidates)
    return data.loc[score.nsmallest(n).index, SHOW].assign(score=score.nsmallest(n).round(3))


example = data.sample(1, random_state=7).index[0]
print("listing the user is looking at:")
display(data.loc[[example], SHOW])
print("recommended listings:")
recommend_similar(example)

In [ ]:
def recommend_for(budget_toman, latitude, longitude, n=5, property_type="apartment-sell"):
    """Listings for a user who gives a budget, a preferred location and a property type."""
    x, y = utm.from_latlon(latitude, longitude, force_zone_number=39)[:2]
    user = pd.DataFrame([[budget_toman, x, y]], columns=FEATURES)
    user_scaled = scaler.transform(user)
    cluster = km_final.predict(user_scaled)[0]
    members = data.index[(data["cluster_final"] == cluster) & (data["cat3_slug"] == property_type)]
    distance = np.linalg.norm(data_scaled.loc[members].to_numpy() - user_scaled, axis=1)
    best = pd.Series(distance, index=members).nsmallest(n)
    print(f"user falls in cluster {cluster}")
    return data.loc[best.index, SHOW].assign(distance=best.round(3))


# a user with 8 billion Toman who wants to live around Vanak, Tehran
recommend_for(8e9, 35.757, 51.410)

<div dir="rtl">

**محدودیت‌ها و قدم بعدی:**
- خوشه‌ها فقط با قیمت و مکان ساخته شده‌اند، چون معیارهای کیفیت خوشه با ویژگی‌های بیشتر بدتر شدند. متراژ در رتبه‌بندی داخل خوشه وارد شده است.
  ویژگی‌هایی مثل آسانسور و پارکینگ را می‌توان به‌صورت فیلتر سخت (نه ویژگی خوشه‌بندی) به این مرحله اضافه کرد.
- قیمت فروش و اجاره مقیاس متفاوتی دارند؛ این سیستم برای آگهی‌های فروش ساخته شده و برای اجاره باید روی «اجاره‌ی معادل» دوباره خوشه‌بندی شود.

</div>

---
# Part 2 — Rent Prediction

A regression model for `rent_value`, trained on log1p of the target.

<div dir="rtl">

**مسیر رسیدن به مدل نهایی** (روی `log1p(rent_value)`):

| مدل | Validation R² |
|---|---|
| Random Forest (مرحله‌ی اولیه) | 0.246 |
| Decision Tree | 0.424 |
| XGBoost پایه | 0.498 |
| XGBoost تنظیم‌شده | 0.558 |
| **XGBoost + مهندسی ویژگی (مدل نهایی)** | **0.612** |

روی داده‌ی Test: **R² = 0.602**، MAE = 0.685 و MSE = 1.240 (هر دو روی مقیاس لگاریتمی). فاصله‌ی Validation و Test حدود ۰٫۰۱ است، یعنی مدل روی داده‌ی Validation بیش‌برازش نکرده است.

**قواعد ارزیابی:** داده با `random_state=42` به ۸۰/۱۰/۱۰ تقسیم شده است. انتخاب ویژگی و پارامترها فقط با Validation انجام شده و Test فقط یک بار، برای مدل نهایی، استفاده شده است.
همه‌ی آمارهای گروهی (فراوانی محله، میانه‌ی متراژ و رهن در محله و شهر) فقط از Train ساخته می‌شوند تا نشت داده رخ ندهد.

</div>

## ۱. بارگذاری داده

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

DIVAR_PATH = Path("../data/Divar.csv")
RANDOM_STATE = 42

COLUMNS = ["cat2_slug", "cat3_slug", "city_slug", "neighborhood_slug", "rent_value", "credit_value",
           "building_size", "rooms_count", "construction_year",
           "has_elevator", "has_parking", "has_warehouse", "has_heating_system"]
df = pd.read_csv(DIVAR_PATH, usecols=COLUMNS, low_memory=False)
print(df.shape)

## ۲. انتخاب آگهی‌های اجاره و پاک‌سازی

<div dir="rtl">

- فقط اجاره‌ی **مسکونی و تجاری** (اجاره‌ی روزانه/موقت جدا است و منطق قیمتش فرق دارد).
- ردیف‌هایی که رهن یا اجاره ندارند و مقادیر ۱۰۰ میلیارد به بالا (اشتباه تایپی) حذف شدند.
- **رهن کل معادل** (`total_credit` = رهن + اجاره × ۱۰۰/۳) فقط برای حذف آگهی‌های بی‌معنی استفاده شد: بیش از ۵۰۰ میلیارد یا کمتر از یک میلیون.
- متراژ غیرممکن **برای هر نوع ملک جداگانه** حذف شد (مثلاً آپارتمان بالای ۵۰۰ متر). متراژهای بزرگ ولی ممکن نگه داشته شدند، چون IQR ساده املاک بزرگ واقعی را هم حذف می‌کرد.

</div>

In [ ]:
rental = df[df["cat2_slug"].isin(["residential-rent", "commercial-rent"])].copy()
print("rental ads:", len(rental))

rental = rental[rental["rent_value"].notna() & rental["credit_value"].notna()
                & (rental["rent_value"] < 1e11) & (rental["credit_value"] < 1e11)]
rental["total_credit"] = rental["credit_value"] + rental["rent_value"] * (100_000_000 / 3_000_000)
rental = rental[rental["total_credit"].between(1e6, 5e11)]

SIZE_LIMITS = {"apartment-rent": 500, "house-villa-rent": 2_000, "shop-rent": 2_000,
               "office-rent": 2_000, "industry-agriculture-business-rent": 500_000}
too_big = rental["building_size"] > rental["cat3_slug"].map(SIZE_LIMITS).fillna(np.inf)
rental = rental[~too_big]
print("after cleaning:", len(rental))

## ۳. متغیر هدف

<div dir="rtl">

- حدود ۱۷٪ آگهی‌ها اجاره‌ی **صفر** دارند. صفر یعنی «رهن کامل» یا «توافقی» و مبلغ واقعی اجاره نیست، پس از مجموعه‌ی هدف حذف شدند.
- `rent_value` به‌شدت راست‌چوله است (میانه ۶٫۳ میلیون، بیشینه ۱۳ میلیارد). مدل روی `log1p(rent_value)` آموزش می‌بیند تا چند اجاره‌ی خیلی بزرگ یادگیری را در دست نگیرند.

</div>

In [ ]:
print("zero rent share:", round((rental["rent_value"] == 0).mean(), 4))
rent = rental[rental["rent_value"] > 0].copy()
print("records used for modelling:", len(rent))
print((rent["rent_value"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]) / 1e6).round(2).rename("rent (million Toman)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(rent["rent_value"] / 1e6, bins=100)
axes[0].set_title("rent_value (million Toman)")
axes[1].hist(np.log1p(rent["rent_value"]), bins=100, color="#27ae60")
axes[1].set_title("log1p(rent_value)")
plt.tight_layout()
plt.show()

## ۴. ویژگی‌های پایه

In [ ]:
FA_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
year = rent["construction_year"].astype("string").str.translate(FA_DIGITS).str.replace("قبل از 1370", "1369", regex=False)
rent["building_age"] = 1405 - pd.to_numeric(year, errors="coerce")

ROOMS = {"بدون اتاق": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج یا بیشتر": 5}
rent["rooms_count_numeric"] = rent["rooms_count"].map(ROOMS)

TRUE_FALSE = {True: 1, "true": 1, "True": 1, False: 0, "false": 0, "False": 0}
for column in ["has_elevator", "has_parking", "has_warehouse"]:
    rent[column] = rent[column].map(TRUE_FALSE).fillna(0)  # not mentioned in the ad -> treated as absent

for column in ["has_heating_system", "cat3_slug", "city_slug", "neighborhood_slug"]:
    rent[column] = rent[column].fillna("Unknown").astype(str)

## ۵. تقسیم Train / Validation / Test

In [ ]:
train, temp = train_test_split(rent, test_size=0.20, random_state=RANDOM_STATE)
validation, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE)
train, validation, test = train.copy(), validation.copy(), test.copy()
print("train:", len(train), " validation:", len(validation), " test:", len(test))

# missing age / rooms are filled with the TRAIN median only
for column in ["building_age", "rooms_count_numeric"]:
    median = train[column].median()
    for part in (train, validation, test):
        part[column] = part[column].fillna(median)

## ۶. مهندسی ویژگی (آمارها فقط از Train)

<div dir="rtl">

| ویژگی | معنی |
|---|---|
| `credit_value_per_size` | رهن به ازای هر متر، سطح قیمت ملک مستقل از اندازه |
| `neighborhood_frequency` | تعداد آگهی‌های محله در Train، نشانه‌ی پرتقاضا بودن محله |
| `neighborhood_share_in_city` | سهم محله از آگهی‌های شهر؛ به‌تنهایی حدود ۰٫۰۰۹ به R² اضافه کرد |
| `building_size_neighborhood_ratio` | متراژ نسبت به متراژ معمول محله |
| `credit_value_neighborhood_median_ratio` | رهن نسبت به رهن معمول محله |
| `credit_value_city_median_ratio` | رهن نسبت به رهن معمول شهر |

مجموع این ویژگی‌ها Validation R² را از ۰٫۵۹۱ به ۰٫۶۱۲ رساند. ویژگی‌هایی مثل `city_type`، `luxury_score` و `has_cooling_system` هم آزمایش شدند ولی بهبودی ندادند و کنار گذاشته شدند.

</div>

In [ ]:
def add_engineered_features(part, reference):
    """Add ratio / frequency features; every group statistic comes from `reference` (the train split)."""
    part = part.copy()
    size = part["building_size"].replace(0, np.nan)
    part["credit_value_per_size"] = part["credit_value"] / size

    neighborhood_count = reference["neighborhood_slug"].value_counts()
    city_count = reference["city_slug"].value_counts()
    part["neighborhood_frequency"] = part["neighborhood_slug"].map(neighborhood_count)
    part["neighborhood_share_in_city"] = part["neighborhood_frequency"] / part["city_slug"].map(city_count)

    size_by_neighborhood = reference.groupby("neighborhood_slug")["building_size"].median()
    credit_by_neighborhood = reference.groupby("neighborhood_slug")["credit_value"].median()
    credit_by_city = reference.groupby("city_slug")["credit_value"].median()
    part["building_size_neighborhood_ratio"] = size / part["neighborhood_slug"].map(size_by_neighborhood).replace(0, np.nan)
    part["credit_value_neighborhood_median_ratio"] = part["credit_value"] / part["neighborhood_slug"].map(credit_by_neighborhood).replace(0, np.nan)
    part["credit_value_city_median_ratio"] = part["credit_value"] / part["city_slug"].map(credit_by_city).replace(0, np.nan)
    return part.replace([np.inf, -np.inf], np.nan)


reference = train.copy()
train = add_engineered_features(train, reference)
validation = add_engineered_features(validation, reference)
test = add_engineered_features(test, reference)

## ۷. ماتریس ویژگی‌ها

In [ ]:
CATEGORICAL = ["city_slug", "cat2_slug", "neighborhood_slug", "has_heating_system", "cat3_slug"]
BASE_NUMERIC = ["building_size", "building_age", "rooms_count_numeric", "has_elevator", "has_parking", "has_warehouse"]
CREDIT_NUMERIC = ["credit_value", "credit_value_per_size", "credit_value_neighborhood_median_ratio", "credit_value_city_median_ratio"]
ENGINEERED = ["neighborhood_frequency", "neighborhood_share_in_city", "building_size_neighborhood_ratio"]


def build_matrices(numeric):
    encoder = OneHotEncoder(handle_unknown="ignore")
    encoder.fit(train[CATEGORICAL])
    matrices = [hstack([csr_matrix(part[numeric].to_numpy(dtype=float)), encoder.transform(part[CATEGORICAL])]).tocsr()
                for part in (train, validation, test)]
    names = numeric + list(encoder.get_feature_names_out(CATEGORICAL))
    return matrices, names


NUMERIC = CREDIT_NUMERIC + BASE_NUMERIC + ENGINEERED
(X_train, X_validation, X_test), feature_names = build_matrices(NUMERIC)
y_train, y_validation, y_test = (np.log1p(part["rent_value"]) for part in (train, validation, test))
print(X_train.shape, X_validation.shape, X_test.shape)

## ۸. مدل نهایی: XGBoost

<div dir="rtl">

پارامترها به‌صورت مرحله‌ای و فقط با Validation تنظیم شدند: اول `max_depth` و `learning_rate`، بعد `n_estimators` و `min_child_weight`، و در آخر `gamma`، `subsample` و `colsample_bytree`.
XGBoost از Random Forest و Decision Tree بهتر بود: بوستینگ خطای مدل‌های قبلی را مرحله‌به‌مرحله اصلاح می‌کند و با هزاران ستون تُنُک one-hot (محله‌ها) هم به‌خوبی کار می‌کند.

⏱ آموزش ۳۰۰۰ درخت با عمق ۱۰ روی حدود ۲۳۰ هزار ردیف چند دقیقه طول می‌کشد.

</div>

In [ ]:
XGB_PARAMS = dict(n_estimators=3000, max_depth=10, min_child_weight=5, learning_rate=0.05, gamma=0,
                  subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
                  tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)

model = XGBRegressor(**XGB_PARAMS)
model.fit(X_train, y_train)

## ۹. ارزیابی

In [ ]:
def report(y_true_log, y_pred_log, name):
    """Metrics on the log scale (what the model optimises) and in Toman (what a user sees)."""
    true_toman, pred_toman = np.expm1(y_true_log), np.expm1(y_pred_log)
    return {
        "split": name,
        "R2 (log)": r2_score(y_true_log, y_pred_log),
        "MAE (log)": mean_absolute_error(y_true_log, y_pred_log),
        "MSE (log)": mean_squared_error(y_true_log, y_pred_log),
        "R2 (Toman)": r2_score(true_toman, pred_toman),
        "MAE (million Toman)": mean_absolute_error(true_toman, pred_toman) / 1e6,
        "MSE (Toman^2)": mean_squared_error(true_toman, pred_toman),
        "median abs % error": np.median(np.abs(pred_toman - true_toman) / true_toman) * 100,
    }


validation_pred = model.predict(X_validation)
results = pd.DataFrame([report(y_train, model.predict(X_train), "train"),
                        report(y_validation, validation_pred, "validation")]).set_index("split")
results.round(4)

<div dir="rtl">

Test فقط **یک بار** و بعد از قطعی شدن ویژگی‌ها و پارامترها اجرا می‌شود:

</div>

In [ ]:
test_pred = model.predict(X_test)
results.loc["test"] = pd.Series(report(y_test, test_pred, "test")).drop("split")
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hexbin(y_test, test_pred, gridsize=80, bins="log", cmap="viridis")
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
axes[0].plot(lims, lims, color="red", linewidth=1)
axes[0].set_xlabel("actual log1p(rent)")
axes[0].set_ylabel("predicted log1p(rent)")
axes[0].set_title("Test set: predicted vs actual")

buckets = pd.cut(np.expm1(y_test), [0, 2e6, 5e6, 1e7, 2e7, 5e7, 1e8, np.inf],
                 labels=["<2M", "2-5M", "5-10M", "10-20M", "20-50M", "50-100M", ">100M"])
error = pd.DataFrame({"bucket": buckets, "abs_pct_error": np.abs(np.expm1(test_pred) - np.expm1(y_test)) / np.expm1(y_test) * 100})
error.groupby("bucket", observed=True)["abs_pct_error"].median().plot(kind="bar", ax=axes[1], color="#e67e22")
axes[1].set_ylabel("median absolute % error")
axes[1].set_xlabel("actual rent")
axes[1].set_title("Test error by rent range")
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_names)
grouped = importance.groupby(lambda name: next((c for c in CATEGORICAL if name.startswith(c + "_")), name)).sum()
grouped.sort_values().tail(15).plot(kind="barh", figsize=(9, 6), title="Feature importance (one-hot columns summed per feature)")
plt.tight_layout()
plt.show()

## ۱۰. بدون رهن: مدل بدون `credit_value`

<div dir="rtl">

مهم‌ترین ورودی مدل، **رهن همان آگهی** است. رهن و اجاره دو بخش یک قیمت‌اند و با هم تعیین می‌شوند، پس مدل بالا در واقع به این سؤال جواب می‌دهد:
«با این مشخصات و **این مقدار رهن**، اجاره چقدر است؟». این نشت متغیر هدف نیست، چون رهن پیش از اجاره در آگهی معلوم است، ولی بخش بزرگی از دقت مدل از همین ستون می‌آید.

برای سنجیدن سهم ویژگی‌های خود ملک، همان مدل بدون هیچ ویژگی وابسته به رهن آموزش داده می‌شود. ⏱ این سلول اختیاری است و آموزش آن هم چند دقیقه طول می‌کشد.

</div>

In [ ]:
RUN_WITHOUT_CREDIT = True

if RUN_WITHOUT_CREDIT:
    (Xn_train, Xn_validation, Xn_test), _ = build_matrices(BASE_NUMERIC + ENGINEERED)
    model_no_credit = XGBRegressor(**XGB_PARAMS).fit(Xn_train, y_train)
    comparison = pd.DataFrame([
        report(y_validation, validation_pred, "with credit (validation)"),
        report(y_validation, model_no_credit.predict(Xn_validation), "without credit (validation)"),
    ]).set_index("split")
    display(comparison.round(4))

## ۱۱. جمع‌بندی

<div dir="rtl">

- مدل نهایی، XGBoost روی `log1p(rent_value)` با ویژگی‌های مکانی و نسبتی، به **R² برابر ۰٫۶۰۲ روی Test** رسید و فاصله‌اش با Validation حدود ۰٫۰۱ بود. یعنی تعمیم مدل پایدار است.
- MAE و MSE روی مقیاس لگاریتمی گزارش شده‌اند. معادل تومانی آن‌ها در جدول ارزیابی آمده؛ R² تومانی کمتر است، چون چند اجاره‌ی خیلی بزرگ در MSE تومانی وزن زیادی دارند.
- خطا در اجاره‌های بالای ۱۰۰ میلیون بیشتر است و مدل آن‌ها را **کم‌برآورد** می‌کند. نمونه‌ی این بازه کم است و تعدادی از آن‌ها احتمالاً اشتباه تایپی‌اند (مثل ۱۱٬۱۱۱٬۱۱۱٬۱۱۱).
- مهم‌ترین ویژگی‌ها: رهن، متراژ، شهر و محله. ویژگی‌هایی که ملک را با محله و شهرش مقایسه می‌کنند بیشترین سود را از مهندسی ویژگی دادند.
- مقایسه‌ی مدل بدون `credit_value` نشان می‌دهد چه بخشی از دقت به دانستن مبلغ رهن وابسته است.

</div>

---
# Part 3 — Sale Price Prediction

A regression model for `price_value`, with group-wise imputation and target encoding.

<div dir="rtl">

**خلاصه‌ی مسیر**

1. فقط آگهی‌های دارای `price_value` (فروش) نگه داشته شدند و ۵٪ پایین و بالای توزیع لگاریتم قیمت (قیمت‌های صفر و اشتباه تایپی) حذف شد.
2. داده **قبل از هر پیش‌پردازشی** به Train / Validation / Test (۶۴ / ۱۶ / ۲۰ درصد) تقسیم شد.
3. هر ستون خالی با یک Transformer جدا پر شد که آمارهایش **فقط روی Train** fit می‌شود. ترتیب جایگزینی: میانه یا مد گروه `cat3_slug`، بعد `cat2_slug`، بعد شهر و در آخر مقدار کلی.
   تعداد اتاق و امکانات بر اساس نوع ملک **و** سطح متراژ (کم / متوسط / زیاد) پر می‌شوند.
4. ویژگی‌های جدید: نسبت زیربنا به زمین، فضای آزاد، متراژ هر اتاق، **فاصله تا مرکز شهر** (هاورسین)، تعداد امکانات، سن بنا و تعامل متراژ و سن.
5. شهر و دسته‌ها با **Target Encoding** (با smoothing) کد شدند، چون one-hot برای ۴۰۰ شهر ابعاد را بی‌دلیل بالا می‌برد.
6. مدل **XGBoost** روی `log1p(price)` آموزش دید و Validation فقط برای early stopping استفاده شد.

**نتیجه‌ی اجرای اصلی:** R² روی Validation **0.608**، روی Test **0.606** و روی Train 0.623؛ MAE روی Test حدود **۱٫۳۶ میلیارد تومان**.
نزدیک بودن R² سه مجموعه یعنی مدل بیش‌برازش ندارد.

سلول‌ها خروجی ذخیره‌شده‌ی اجرای اصلی را دارند. سلول‌های بخش آخر (MSE و نمودارها) تازه اضافه شده‌اند و با یک بار اجرا خروجی می‌گیرند.

</div>

## ۱. بارگذاری داده

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

DIVAR_PATH = Path("../data/Divar.csv")
data_frame = pd.read_csv(DIVAR_PATH, low_memory=False)

data_frame.head(10)

## ۲. انتخاب آگهی‌های فروش، حذف پرت‌ها و انتخاب ویژگی‌ها

**We make a copy of the main data frame.**

In [ ]:
cp_data_frame = data_frame.copy()
cp_data_frame["is_rebuilt"].isna().sum()

 **First step is to remove the rows in the price_value column.**

In [ ]:

cp_data_frame = cp_data_frame[cp_data_frame["price_value"].notna()].copy()

price_frame = np.log1p(cp_data_frame["price_value"].astype("float64"))
lower_bound, upper_bound = price_frame.quantile(0.05), price_frame.quantile(0.95)
mask = ((price_frame >= lower_bound) & (price_frame <= upper_bound)).to_numpy()
cp_data_frame = cp_data_frame[mask].copy()


cp_data_frame.head()

**As we know the number of the lines with the Nan city_slug is less than 10 so we remove them.**


In [ ]:
cp_data_frame = cp_data_frame.dropna(subset=["city_slug"])
cp_data_frame["city_slug"].isna()

**In this part we just get the effective features.**

In [ ]:
cp_data_frame = cp_data_frame[["city_slug", "price_value", "land_size", "building_size", "rooms_count", "has_warehouse", "has_parking", "location_latitude"
  , "location_longitude", "cat2_slug", "cat3_slug", "construction_year","has_elevator", "has_balcony"
  ]]



## ۳. تقسیم داده (پیش از هر fit)

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(cp_data_frame, test_size=0.2, random_state=42)
train_data, validation_data = train_test_split(train_data, test_size=0.2, random_state=42)

## ۴. پر کردن مقادیر خالی (fit فقط روی Train)

**As you can see the informations in the rooms_count are in the Persian language so we map the data into English.**

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def room_count_persian(data_frame : pd.DataFrame) -> pd.DataFrame:
  map_rooms_count = {
    "دو" : 2,
    "یک" : 1,
    "سه" : 3,
    "بدون اتاق" : 0,
    "چهار" : 4,
    "پنج یا بیشتر" : 5
  }
  data_frame["rooms_count"] = data_frame["rooms_count"].map(map_rooms_count)
  return data_frame

room_count_transformer = FunctionTransformer(room_count_persian)
train_data = room_count_transformer.transform(train_data)
test_data = room_count_transformer.transform(test_data)
validation_data = room_count_transformer.transform(validation_data)


train_data.head()


**We group the data base on the cat3_slug , cat3_slug and city_slug. Then we fill the Nan values respectivly. This logic has been used for the rest. In some cases we used median or mode of the each group.**

*   1- > cat3_slug
*   2-> cat2_slug
* 3-> city_slug



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

class fill_constrution(BaseEstimator, TransformerMixin):
    def __init__(self, target_col: str = "construction_year", downcast: bool = True):
        self.target_col = target_col
        self.downcast = downcast
        self.cat3_medians_ = {}
        self.cat2_medians_ = {}
        self.city_medians_ = {}
        self.global_median_ = None

    def fit(self, X: pd.DataFrame, y=None):
        self.global_median_ = X[self.target_col].median()
        self.city_medians_ = X.groupby("city_slug")[self.target_col].median().to_dict()
        self.cat2_medians_ = X.groupby("cat2_slug")[self.target_col].median().to_dict()
        self.cat3_medians_ = X.groupby("cat3_slug")[self.target_col].median().to_dict()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()

        imputed_cat3 = df["cat3_slug"].map(self.cat3_medians_)
        imputed_cat2 = df["cat2_slug"].map(self.cat2_medians_)
        imputed_city = df["city_slug"].map(self.city_medians_)

        df[self.target_col] = (
            df[self.target_col]
            .fillna(imputed_cat3)
            .fillna(imputed_cat2)
            .fillna(imputed_city)
            .fillna(self.global_median_)
        )

        if self.downcast:
            df[self.target_col] = df[self.target_col].round().astype("int16")

        return df

def clean_fa_digits(series: pd.Series) -> pd.Series:
    fa_digits = "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩"
    en_digits = "01234567890123456789"
    trans_table = str.maketrans(fa_digits, en_digits)
    return series.astype(str).str.translate(trans_table)


def reshape_construction_year(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    cleaned_col = clean_fa_digits(data["construction_year"])
    cleaned_col = cleaned_col.replace(
        {
            "قبل از 1370": "1369",
            "قبل از ۱۳۷۰": "1369",
        }
    )

    data["construction_year"] = pd.to_numeric(cleaned_col, errors="coerce")
    return data


reshaper = FunctionTransformer(reshape_construction_year)
train_data = reshaper.transform(train_data)
test_data = reshaper.transform(test_data)
validation_data = reshaper.transform(validation_data)


imputer = fill_constrution(target_col="construction_year", downcast=True)
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


train_data["construction_year"]


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_building_size(BaseEstimator, TransformerMixin):
    def __init__(self, group_col: str = "city_slug", target_col: str = "building_size", downcast: bool = True):
        self.group_col = group_col
        self.target_col = target_col
        self.downcast = downcast
        self.group_medians_3 = {}
        self.group_medians_2 = {}
        self.group_medians_1 = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        # calculating global median.
        self.global_median_ = data_frame[self.target_col].median()

        # calculating the median of each group based in the city_slug.
        self.group_medians_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .median()
            .to_dict()
        )
        return self

    def transform(self, data_frame, y=None):
        data_frame = data_frame.copy()
        missing_mask = data_frame[self.target_col].isna()

        if missing_mask.sum() == 0:
            return data_frame

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_medians_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_medians_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_medians_1)

        imputed_value = (
            group_imputed_3
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_median_)
        )

        data_frame.loc[missing_mask, self.target_col] = imputed_value[missing_mask]
        return data_frame

building_imputer = fill_building_size(group_col="city_slug", target_col="building_size", downcast=True)

train_data = building_imputer.fit_transform(train_data)
test_data = building_imputer.transform(test_data)
validation_data = building_imputer.transform(validation_data)


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_land_size(BaseEstimator, TransformerMixin):
    def __init__(self, group_col: str = "city_slug", target_col: str = "land_size", downcast: bool = True):
        self.group_col = group_col
        self.target_col = target_col
        self.downcast = downcast
        self.group_medians_3 = {}
        self.group_medians_2 = {}
        self.group_medians_1 = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        # global median
        self.global_median_ = data_frame[self.target_col].median()

        # grouped median based on the city_slug.
        self.group_medians_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .median()
            .to_dict()
        )
        self.group_medians_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .median()
            .to_dict()
        )
        return self

    def transform(self, data_frame, y=None):
        data_frame = data_frame.copy()
        missing_mask = data_frame[self.target_col].isna()

        if missing_mask.sum() == 0:
            return data_frame

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_medians_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_medians_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_medians_1)

        imputed_value = (
            group_imputed_3
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_median_)
        )

        data_frame.loc[missing_mask, self.target_col] = imputed_value[missing_mask]
        return data_frame

imputer = fill_land_size(group_col="city_slug", target_col="land_size", downcast=True)

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


**This cell is used to label the rows based on the building size**

In [ ]:
def building_size_patintion(data_frame : pd.DataFrame):
    df = data_frame.copy()

    df.loc[df["building_size"] < 150, "building_size_level"] = "Low"
    df.loc[(df["building_size"] >= 150) & (df["building_size"] < 300), "building_size_level"] = "Medium"
    df.loc[df["building_size"] >= 300, "building_size_level"] = "High"

    return df

building_size_patintion = FunctionTransformer(building_size_patintion)
train_data = building_size_patintion.transform(train_data)
test_data = building_size_patintion.transform(test_data)
validation_data = building_size_patintion.transform(validation_data)


train_data.columns

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_room_count(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "rooms_count", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )

        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )

        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        keys_3 = list(
            zip(data_frame["cat3_slug"], data_frame["building_size_level"])
        )
        mode_imputed_3 = pd.Series(keys_3, index=data_frame.index).map(
            self.group_mode_3
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            mode_imputed_3
        )

        if data_frame[self.target_col].isna().any():
            keys_2 = list(
                zip(data_frame["cat2_slug"], data_frame["building_size_level"])
            )
            mode_imputed_2 = pd.Series(keys_2, index=data_frame.index).map(
                self.group_mode_2
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_2
            )

        if data_frame[self.target_col].isna().any():
            keys_1 = list(
                zip(data_frame["city_slug"], data_frame["building_size_level"])
            )
            mode_imputed_1 = pd.Series(keys_1, index=data_frame.index).map(
                self.group_mode_1
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_1
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

imputer = fill_room_count()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)



**The normalize_bool_col is used to turn booleans to inteegers**
1.   True : 1
2.   False : 0



In [ ]:
def normalize_bool_col(series: pd.Series) -> pd.Series:
    mapping = {
        True: 1, False: 0,
        'True': 1, 'False': 0,
        'true': 1, 'false': 0,
        '1': 1, '0': 0,
        1: 1, 0: 0,
    }
    return series.map(mapping)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class fill_elavator(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_elevator", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )

        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )

        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        keys_3 = list(
            zip(data_frame["cat3_slug"], data_frame["building_size_level"])
        )
        mode_imputed_3 = pd.Series(keys_3, index=data_frame.index).map(
            self.group_mode_3
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            mode_imputed_3
        )

        if data_frame[self.target_col].isna().any():
            keys_2 = list(
                zip(data_frame["cat2_slug"], data_frame["building_size_level"])
            )
            mode_imputed_2 = pd.Series(keys_2, index=data_frame.index).map(
                self.group_mode_2
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_2
            )

        if data_frame[self.target_col].isna().any():
            keys_1 = list(
                zip(data_frame["city_slug"], data_frame["building_size_level"])
            )
            mode_imputed_1 = pd.Series(keys_1, index=data_frame.index).map(
                self.group_mode_1
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                mode_imputed_1
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

train_data['has_elevator'] = normalize_bool_col(train_data['has_elevator'])
test_data['has_elevator'] = normalize_bool_col(test_data['has_elevator'])
validation_data['has_elevator'] = normalize_bool_col(validation_data['has_elevator'])



imputer = fill_elavator()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class fill_balcany(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_balcony", dtype: str = "int8"):
        self.target_col = target_col
        self.dtype = dtype
        self.group_mode_3 = {}
        self.group_mode_2 = {}
        self.group_mode_1 = {}
        self.global_median_ = None

    def _calc_mode(
        self, df: pd.DataFrame, group_cols: list[str]
    ) -> dict[tuple, float]:
        mode_series = (
            df.groupby(group_cols)[self.target_col]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .dropna()
        )
        return mode_series.to_dict()

    def fit(self, data_frame: pd.DataFrame, y=None):
        df = data_frame.copy()
        df[self.target_col] = normalize_bool_col(df[self.target_col])

        target_series = df[self.target_col]

        valid_mask = target_series.notna()
        df_valid = df.loc[valid_mask].copy()

        self.group_mode_3 = self._calc_mode(
            df_valid, ["cat3_slug", "building_size_level"]
        )
        self.group_mode_2 = self._calc_mode(
            df_valid, ["cat2_slug", "building_size_level"]
        )
        self.group_mode_1 = self._calc_mode(
            df_valid, ["city_slug", "building_size_level"]
        )

        self.global_median_ = df_valid[self.target_col].median()

        return self


    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        data_frame[self.target_col] = pd.to_numeric(
            data_frame[self.target_col], errors="coerce"
        )

        index_3 = pd.MultiIndex.from_arrays(
            [data_frame["cat3_slug"], data_frame["building_size_level"]]
        )
        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            pd.Series(index_3.map(self.group_mode_3), index=data_frame.index)
        )

        if data_frame[self.target_col].isna().any():
            index_2 = pd.MultiIndex.from_arrays(
                [data_frame["cat2_slug"], data_frame["building_size_level"]]
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                pd.Series(
                    index_2.map(self.group_mode_2), index=data_frame.index
                )
            )

        if data_frame[self.target_col].isna().any():
            index_1 = pd.MultiIndex.from_arrays(
                [data_frame["city_slug"], data_frame["building_size_level"]]
            )
            data_frame[self.target_col] = data_frame[self.target_col].fillna(
                pd.Series(
                    index_1.map(self.group_mode_1), index=data_frame.index
                )
            )

        data_frame[self.target_col] = data_frame[self.target_col].fillna(
            self.global_median_
        )

        data_frame[self.target_col] = np.round(
            data_frame[self.target_col]
        ).astype(self.dtype)

        return data_frame

imputer = fill_balcany()
train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


In this part we want to fill the has_parking.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


train_data["has_parking"] = normalize_bool_col(train_data["has_parking"])
test_data["has_parking"] = normalize_bool_col(test_data["has_parking"])
validation_data["has_parking"] = normalize_bool_col(validation_data["has_parking"])


class fill_has_parking(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_parking"):
        self.target_col = target_col
        self.group_modes_3 = {}
        self.group_modes_2 = {}
        self.group_modes_1 = {}
        self.global_mode_ = None

    @staticmethod
    def _calc_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else np.nan

    def fit(self, data_frame: pd.DataFrame, y=None):

        m = data_frame[self.target_col].mode()
        self.global_mode_ = m.iloc[0] if not m.empty else np.nan

        self.group_modes_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_modes_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_modes_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_modes_1)

        data_frame[self.target_col] = (
            data_frame[self.target_col]
            .fillna(group_imputed_3)
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_mode_)
        )

        return data_frame



imputer = fill_has_parking(target_col="has_parking")

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


print(train_data["has_parking"].value_counts(dropna=False))


In [ ]:
class fill_cordinate(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        target_col: str,
        city_col: str = "city_slug",
        dtype: str = "float64"
    ):
        self.target_col = target_col
        self.city_col = city_col
        self.dtype = dtype

        self.city_median_ = {}
        self.global_median_ = None

    def fit(self, data_frame: pd.DataFrame, y=None):
        valid_mask = data_frame[self.target_col].notna()
        df_valid = data_frame.loc[valid_mask]

        # calculating the medain based on the city.
        self.city_median_ = (
            df_valid.groupby(self.city_col)[self.target_col]
            .median()
            .dropna()
            .to_dict()
        )

        # global median.
        self.global_median_ = df_valid[self.target_col].median()

        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        # based on the city.
        imputed_l2 = data_frame[self.city_col].map(self.city_median_)
        data_frame[self.target_col] = data_frame[self.target_col].fillna(imputed_l2)

        # global median.
        data_frame[self.target_col] = data_frame[self.target_col].fillna(self.global_median_)

        # Casting
        data_frame[self.target_col] = data_frame[self.target_col].astype(self.dtype)

        return data_frame


lat_imputer = fill_cordinate(
    target_col="location_latitude",
    city_col="city_slug",
    dtype="float64"
)

lon_imputer = fill_cordinate(
    target_col="location_longitude",
    city_col="city_slug",
    dtype="float64"
)

train_data = lat_imputer.fit_transform(train_data)
train_data = lon_imputer.fit_transform(train_data)

validation_data = lat_imputer.transform(validation_data)
validation_data = lon_imputer.transform(validation_data)

test_data = lat_imputer.transform(test_data)
test_data = lon_imputer.transform(test_data)



In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

train_data['has_warehouse'] = normalize_bool_col(train_data['has_warehouse'])
test_data['has_warehouse'] = normalize_bool_col(test_data['has_warehouse'])


class fill_has_warehouse(BaseEstimator, TransformerMixin):

    def __init__(self, target_col: str = "has_warehouse"):
        self.target_col = target_col
        self.group_modes_3 = {}
        self.group_modes_2 = {}
        self.group_modes_1 = {}
        self.global_mode_ = None

    @staticmethod
    def _calc_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else np.nan

    def fit(self, data_frame: pd.DataFrame, y=None):

        m = data_frame[self.target_col].mode()
        self.global_mode_ = m.iloc[0] if not m.empty else np.nan

        self.group_modes_3 = (
            data_frame.groupby("cat3_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_2 = (
            data_frame.groupby("cat2_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        self.group_modes_1 = (
            data_frame.groupby("city_slug")[self.target_col]
            .agg(self._calc_mode)
            .to_dict()
        )
        return self

    def transform(self, data_frame: pd.DataFrame) -> pd.DataFrame:
        data_frame = data_frame.copy()

        group_imputed_3 = data_frame["cat3_slug"].map(self.group_modes_3)
        group_imputed_2 = data_frame["cat2_slug"].map(self.group_modes_2)
        group_imputed_1 = data_frame["city_slug"].map(self.group_modes_1)

        data_frame[self.target_col] = (
            data_frame[self.target_col]
            .fillna(group_imputed_3)
            .fillna(group_imputed_2)
            .fillna(group_imputed_1)
            .fillna(self.global_mode_)
        )

        return data_frame


imputer = fill_has_warehouse(target_col="has_warehouse")

train_data = imputer.fit_transform(train_data)
test_data = imputer.transform(test_data)
validation_data = imputer.transform(validation_data)


print(train_data["has_warehouse"].value_counts(dropna=False))


## ۵. کدگذاری و مهندسی ویژگی

**In this part label the data based on the building_level which wa created before.
Also we want to the categorical feature in our data frame.**

In [ ]:
cat_cols = ["city_slug", "cat2_slug", "cat3_slug"]

categories = {
    c: pd.Index(train_data[c].dropna().unique())
    for c in cat_cols
}

for c in cat_cols:
    train_data[c] = pd.Categorical(train_data[c], categories=categories[c])
    test_data[c]  = pd.Categorical(test_data[c],  categories=categories[c])
    validation_data[c] = pd.Categorical(validation_data[c], categories=categories[c])

level_map = {"Low": 1, "Medium": 2, "High": 3}
train_data["building_size_level"] = (
    train_data["building_size_level"].map(level_map).astype("UInt8")
)
test_data["building_size_level"] = (
    test_data["building_size_level"].map(level_map).astype("UInt8")
)
validation_data["building_size_level"] = (
    validation_data["building_size_level"].map(level_map).astype("UInt8")
)


train_data = train_data.dropna(subset=["price_value"])
test_data = test_data.dropna(subset=["price_value"])

train_x = train_data.drop(columns=["price_value"])
test_x = test_data.drop(columns=["price_value"])
val_x = validation_data.drop(columns=["price_value"])

train_y = train_data["price_value"]
test_y = test_data["price_value"]
val_y = validation_data["price_value"]


In [ ]:
city_lat_center = train_x.groupby('city_slug')['location_latitude'].median()
city_lon_center = train_x.groupby('city_slug')['location_longitude'].median()


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_features(df, city_lat_center, city_lon_center):
    df = df.copy()

    df['building_land_ratio'] = df['building_size'] / (df['land_size'] + 1)
    df["free_area"] = df["land_size"]  - df["building_size"]
    df['area_per_room'] = df['building_size'] / df['rooms_count'].replace(0, 1)

    center_lat = df['city_slug'].map(city_lat_center).astype(float)
    center_lon = df['city_slug'].map(city_lon_center).astype(float)
    lat = df['location_latitude'].astype(float).values
    lon = df['location_longitude'].astype(float).values
    df['dist_to_city_center'] = haversine_distance(
        df['location_latitude'].values,
        df['location_longitude'].values,
        center_lat.values,
        center_lon.values,
    )
    df['amenity_count'] = (
    df['has_warehouse'].astype(int) +
    df['has_parking'].astype(int) +
    df['has_elevator'].astype(int) +
    df['has_balcony'].astype(int)
    )

    df['building_age'] = (1403 - df['construction_year']).clip(0, 100)

    df['size_age_interact'] = df['building_size'] / (df['building_age'] + 1)

    label = (df["building_age"] <= 10)
    df[label]["is_newly_built"] = 1
    df[~label]["is_newly_built"] = 0

    new_cols = [
        'building_land_ratio',
        'area_per_room',
        'dist_to_city_center',
        'building_age',
        'size_age_interact',
        "free_area",
        "amenity_count"
    ]
    for col in new_cols:
        df[col] = df[col].fillna(df[col].median())

    return df

train_x = add_features(train_x, city_lat_center, city_lon_center)
val_x = add_features(val_x, city_lat_center, city_lon_center)
test_x = add_features(test_x, city_lat_center, city_lon_center)



**Give a code to each category in the data frame bases on its weigth to the target.**

In [ ]:
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['city_slug','cat2_slug','cat3_slug'], smoothing=10)
train_x = encoder.fit_transform(train_x, train_y)
val_x   = encoder.transform(val_x)
test_x   = encoder.transform(test_x)

In [ ]:
val_x['has_parking'] = normalize_bool_col(val_x['has_parking']).fillna(0).astype('int8')
test_x['has_parking'] = normalize_bool_col(test_x['has_parking']).fillna(0).astype('int8')
train_x['has_parking'] = normalize_bool_col(train_x['has_parking']).fillna(0).astype('int8')

train_x['has_warehouse'] = normalize_bool_col(train_x['has_warehouse'])
val_x['has_warehouse']   = normalize_bool_col(val_x['has_warehouse'])
test_x['has_warehouse']  = normalize_bool_col(test_x['has_warehouse'])

object_cols = train_x.select_dtypes(include='object').columns.tolist()
if object_cols:
    print("Still object dtype:", object_cols)
else:
    print("All dtypes OK — ready for XGBoost")


## ۶. مدل XGBoost

In this case we use a xgboost to test our preproccessing.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import numpy as np

train_y = np.log1p(train_y)
test_y  = np.log1p(test_y)
val_y   = np.log1p(val_y)

model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    gamma=0.1,
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=100,
    eval_metric='rmse',
    n_jobs=-1,
    enable_categorical=True,
)

model.fit(
    train_x, train_y,
    eval_set=[(val_x, val_y)],
    verbose=200
)

y_pred_log = model.predict(val_x)
y_pred = np.expm1(y_pred_log)

val_y = np.expm1(val_y)
train_y = np.expm1(train_y)
test_y = np.expm1(test_y)

print(f"MAE    : {mean_absolute_error(val_y, y_pred):,.0f}")
print(f"R2     : {r2_score(val_y, y_pred):.4f}")

train_pred = np.expm1(model.predict(train_x))
print(f"Train R2: {r2_score(train_y, train_pred):.4f}")

test_pred = np.expm1(model.predict(test_x))
print(f"MAE Test    : {mean_absolute_error(test_y, test_pred):,.0f}")
print(f"R2 Test     : {r2_score(test_y, test_pred):.4f}")


## ۷. گزارش کامل معیارها (R²، MAE، MSE)

<div dir="rtl">

صورت پروژه R²، MAE و MSE را روی داده‌ی Test می‌خواهد. سلول بالا فقط R² و MAE را چاپ می‌کرد؛ این‌جا هر سه معیار، هم به تومان و هم روی مقیاس لگاریتمی که مدل با آن آموزش دیده، گزارش می‌شوند.

</div>

In [ ]:
from sklearn.metrics import mean_squared_error


def report(y_true, y_pred, name):
    return {
        "split": name,
        "R2": r2_score(y_true, y_pred),
        "MAE (billion Toman)": mean_absolute_error(y_true, y_pred) / 1e9,
        "MSE (Toman^2)": mean_squared_error(y_true, y_pred),
        "RMSE (billion Toman)": np.sqrt(mean_squared_error(y_true, y_pred)) / 1e9,
        "R2 (log)": r2_score(np.log1p(y_true), np.log1p(y_pred)),
        "MAE (log)": mean_absolute_error(np.log1p(y_true), np.log1p(y_pred)),
        "MSE (log)": mean_squared_error(np.log1p(y_true), np.log1p(y_pred)),
        "median abs % error": np.median(np.abs(y_pred - y_true) / y_true) * 100,
    }


metrics = pd.DataFrame([report(train_y, train_pred, "train"),
                        report(val_y, y_pred, "validation"),
                        report(test_y, test_pred, "test")]).set_index("split")
metrics.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hexbin(np.log10(test_y), np.log10(test_pred), gridsize=70, bins="log", cmap="viridis")
lims = [np.log10(test_y).min(), np.log10(test_y).max()]
axes[0].plot(lims, lims, color="red", linewidth=1)
axes[0].set_xlabel("actual price (log10 Toman)")
axes[0].set_ylabel("predicted price (log10 Toman)")
axes[0].set_title("Test set: predicted vs actual")

importance = pd.Series(model.feature_importances_, index=train_x.columns).sort_values()
importance.tail(15).plot(kind="barh", ax=axes[1], color="#2471a3")
axes[1].set_title("XGBoost feature importance (gain share)")
plt.tight_layout()
plt.show()

## ۸. جمع‌بندی

<div dir="rtl">

- مدل نهایی حدود **۶۱٪ واریانس قیمت** را روی داده‌ی دیده‌نشده توضیح می‌دهد. نزدیکی R² در Train، Validation و Test نشان می‌دهد مدل بیش‌برازش نکرده است.
- قیمت فروش در این داده ترکیبی از آپارتمان، ویلا، زمین، پیش‌فروش و ملک تجاری در صدها شهر است و ویژگی‌های مهمی مثل کیفیت ساخت، دقیق بودن موقعیت و سند در آن نیست. برای همین سقف دقیقی که می‌شود انتظار داشت محدود است.
- MAE حدود ۱٫۳ میلیارد تومان در برابر میانه‌ی قیمت حدود ۳ میلیارد زیاد به نظر می‌رسد، ولی خطا نسبی است: در ملک‌های ارزان کم و در ملک‌های گران زیاد است. معیار «میانه‌ی درصد خطا» در جدول بالا تصویر واقعی‌تری می‌دهد.
- **ریسک ارزیابی:** ۱۰٪ دو سر توزیع قیمت، قبل از تقسیم و برای Test هم، حذف شده است. پس معیارها برای قیمت‌های «معمول» معتبرند و نه برای ملک‌های خیلی ارزان یا خیلی گران.
- قدم‌های بعدی: مدل جدا برای هر نوع ملک (آپارتمان، ویلا، زمین)، استفاده از محله با Target Encoding، و تنظیم پارامترها با جست‌وجوی سیستماتیک روی Validation.

</div>